<a href="https://colab.research.google.com/github/JaimRM/QuantitativeFinance/blob/main/Predicci%C3%B3n_Bankinter_precio_volumen_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install tensorflow

In [2]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler

# 1. Datos: [Precio de cierre, Volumen en millones]
# Datos simulados coherentes con la tendencia de Bankinter
datos_bkt = np.array([
    [13.32, 2.1], [13.38, 2.5], [13.37, 1.8], [13.83, 3.2],
    [13.64, 2.0], [13.67, 2.1], [14.40, 4.5], [14.26, 3.0],
    [14.49, 3.5], [14.60, 3.8], [14.90, 4.2], [14.88, 3.9]
])

# 2. Normalización de ambas columnas por separado
scaler = MinMaxScaler()
datos_norm = scaler.fit_transform(datos_bkt)

# 3. Crear ventanas (X incluirá Precio y Volumen, y queremos predecir solo Precio)
def crear_ventanas_multi(datos, ventana=3):
    X, y = [], []
    for i in range(len(datos) - ventana):
        X.append(datos[i:i+ventana, :]) # Todas las columnas (Precio y Vol)
        y.append(datos[i+ventana, 0])    # Solo la columna 0 (Precio)
    return np.array(X), np.array(y)

X, y = crear_ventanas_multi(datos_norm)

# 4. Modelo LSTM
model = Sequential([
    # input_shape ahora es (3 pasos, 2 características)
    LSTM(50, activation='relu', input_shape=(3, 2)),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')

# 5. Entrenamiento
model.fit(X, y, epochs=600, verbose=0)

# 6. Predicción para mañana
ultima_ventana = datos_norm[-3:].reshape(1, 3, 2)
prediccion_norm = model.predict(ultima_ventana, verbose=0)

# Para des-normalizar el precio, necesitamos un "truco"
# (el scaler espera 2 columnas, así que creamos una fila ficticia)
dummy = np.zeros((1, 2))
dummy[0, 0] = prediccion_norm.item()
prediccion_final = scaler.inverse_transform(dummy)[0, 0]

print(f"Predicción Bankinter (usando Precio+Volumen): {prediccion_final:.3f}€")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Predicción Bankinter (usando Precio+Volumen): 15.203€


In [4]:

from tensorflow.keras.layers import Input
# 1. Modificamos la función para que guarde los dos datos en 'y'
def crear_ventanas_multi_salida(datos, ventana=3):
    X, y = [], []
    for i in range(len(datos) - ventana):
        X.append(datos[i:i+ventana, :]) # Mira los 3 días anteriores (Precio y Vol)
        y.append(datos[i+ventana, :])    # ¡NUEVO! Guarda AMBOS datos para el día siguiente
    return np.array(X), np.array(y)

X, y = crear_ventanas_multi_salida(datos_norm)

# 2. Modificamos el modelo
model = Sequential([
    Input(shape=(3, 2)),
    LSTM(50, activation='relu'),
    Dense(2) # ¡NUEVO! Cambiamos a 2 para que devuelva [Precio, Volumen]
])
model.compile(optimizer='adam', loss='mse')

# 3. Entrenamiento (igual que antes)
model.fit(X, y, epochs=600, verbose=0)

# 4. Predicción sin trucos
ultima_ventana = datos_norm[-3:].reshape(1, 3, 2)
prediccion_norm = model.predict(ultima_ventana, verbose=0)

# Como 'prediccion_norm' ya viene con 2 columnas [Precio, Volumen],
# el scaler lo procesa directamente sin 'dummy'
prediccion_final = scaler.inverse_transform(prediccion_norm)

precio_mañana = prediccion_final[0, 0]
volumen_mañana = prediccion_final[0, 1]

print(f"Predicción Bankinter para mañana:")
print(f"-> Precio: {precio_mañana:.3f}€")
print(f"-> Volumen: {volumen_mañana:.2f} millones de acciones")

Predicción Bankinter para mañana:
-> Precio: 15.048€
-> Volumen: 4.31 millones de acciones
